In [0]:
# Databricks notebook source
# =============================================================
# nb_bronze_to_silver.py
# Layer     : Silver
# Purpose   : Orchestrator only. Coordinates the 3 specialist
#             notebooks. Contains zero business logic itself.
#
# Called by : ADF Notebook Activity (inside ForEach loop)
# Calls     : set_auth_context → silver_reader →
#             cleaner → silver_writer
# =============================================================

In [0]:
# Cell 1: Widget setup
def setup_widgets():
    print("[INFO] Ensuring widgets are available...")
    widget_defaults = {
        "table_name": "device_snapshots",
        "table_type": "fact",
        "load_date": "2025-01-01",
        "storage_account": "petiot",
        "is_incremental": "true",
    }

    for name, default in widget_defaults.items():
        try:
            current_value = dbutils.widgets.get(name).strip()
            if current_value:
                print(f"[INFO] Preserving existing widget: {name}={current_value}")
                continue
        except Exception:
            pass

        dbutils.widgets.text(name, default)
        print(f"[INFO] Created widget: {name}={default}")

    print("[INFO] Widget setup complete.")

In [0]:
# Cell 2: Parameter reading
def get_params():
    print("[INFO] Reading widget parameters...")
    TABLE_NAME = dbutils.widgets.get("table_name").strip()
    
    # --- ADD THIS SAFEGUARD FROM YOUR METADATA JSON ---
    # List of tables explicitly configured as dimensions in your JSON
    DIMENSION_TABLES = ["owners", "pets", "products", "firmware", "batches", "devices"]
    
    if TABLE_NAME in DIMENSION_TABLES:
        print(f"[SAFEGUARD] Auto-correcting metadata parameters for Dimension table: '{TABLE_NAME}'")
        TABLE_TYPE = "dimension"
        IS_INCREMENTAL = False
    else:
        # Fall back to standard widget values for fact tables
        TABLE_TYPE = dbutils.widgets.get("table_type").strip()
        IS_INCREMENTAL = dbutils.widgets.get("is_incremental").strip().lower() == "true"
    # --------------------------------------------------

    LOAD_DATE = dbutils.widgets.get("load_date").strip()
    STORAGE_ACCOUNT = dbutils.widgets.get("storage_account").strip()

    print(f"[INFO] Final active execution configurations:")
    print(f"       TABLE_NAME      = {TABLE_NAME}")
    print(f"       TABLE_TYPE      = {TABLE_TYPE}")
    print(f"       IS_INCREMENTAL  = {IS_INCREMENTAL}")
    print(f"       LOAD_DATE       = {LOAD_DATE}")
    
    return TABLE_NAME, TABLE_TYPE, LOAD_DATE, STORAGE_ACCOUNT, IS_INCREMENTAL

In [0]:
# Cell 3: Header printing
def print_header(TABLE_NAME, TABLE_TYPE, LOAD_DATE, STORAGE_ACCOUNT, IS_INCREMENTAL):
    print("=" * 60)
    print(f"  nb_bronze_to_silver")
    print(f"  Table         : {TABLE_NAME}")
    print(f"  Type          : {TABLE_TYPE}")
    print(f"  Load date     : {LOAD_DATE}")
    print(f"  Storage       : {STORAGE_ACCOUNT}")
    print(f"  Incremental   : {IS_INCREMENTAL}")
    print("=" * 60)

In [0]:
# Cell 4: Notebook runner and path utils
import json

def run_notebook(notebook_path: str) -> None:
    print(f"[INFO] Loading notebook into current session: {notebook_path}")
    try:
        get_ipython().run_line_magic("run", notebook_path)
        print(f"[INFO] Notebook loaded: {notebook_path}")
    except Exception as e:
        print(f"[ERROR] Failed to load notebook: {notebook_path}\n{e}")
        raise


def log_step(message: str) -> None:
    print(f"[nb_bronze_to_silver] {message}")


def abfss_path(container: str, path: str = "", storage_account: str | None = None) -> str:
    active_storage_account = storage_account or STORAGE_ACCOUNT
    clean_path = path.lstrip("/")
    base = f"abfss://{container}@{active_storage_account}.dfs.core.windows.net"
    return f"{base}/{clean_path}" if clean_path else f"{base}/"


def abfss(container: str, path: str = "", storage_account: str | None = None) -> str:
    return abfss_path(container, path, storage_account)


def ensure_auth_context() -> None:
    auth_type_key = f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net"
    if spark.conf.get(auth_type_key, None):
        log_step("Azure auth context already available.")
        return

    log_step("Azure auth context missing. Loading shared auth notebook.")
    run_notebook("/Workspace/Shared/pet-analytics/shared/set_auth_context.py.ipynb")
    log_step("Azure auth context loaded.")

In [0]:
%run /Workspace/Shared/pet-analytics/silver/silver_reader

In [0]:
%run /Workspace/Shared/pet-analytics/silver/cleaner

In [0]:
%run /Workspace/Shared/pet-analytics/silver/silver_writer

In [0]:
# Debug helper: inspect Bronze/Silver coverage for the active widget values
from pyspark.sql import functions as F

def debug_current_run_inputs():
    storage_account = dbutils.widgets.get("storage_account")
    table_name = dbutils.widgets.get("table_name")
    load_date = dbutils.widgets.get("load_date")

    bronze_path = abfss("bronze", table_name, storage_account)
    silver_path = abfss("silver", table_name, storage_account)
    raw_path = abfss("raw", table_name, storage_account)

    print("=" * 80)
    print("[debug] Current run inputs")
    print(f"[debug] storage_account = {storage_account}")
    print(f"[debug] table_name      = {table_name}")
    print(f"[debug] load_date       = {load_date}")
    print(f"[debug] raw_path        = {raw_path}")
    print(f"[debug] bronze_path     = {bronze_path}")
    print(f"[debug] silver_path     = {silver_path}")
    print("=" * 80)

    bronze_df = spark.read.format("delta").load(bronze_path)
    print(f"[debug] Bronze total rows: {bronze_df.count():,}")

    if "_load_date" in bronze_df.columns:
        normalized_load_date = load_date if load_date.startswith("year=") else f"year={load_date[:4]}/month={load_date[5:7]}/day={load_date[8:10]}"
        print(f"[debug] Normalized Bronze _load_date = {normalized_load_date}")
        print(f"[debug] Bronze rows for widget date  = {bronze_df.filter(F.col('_load_date') == normalized_load_date).count():,}")
        print("[debug] Latest Bronze _load_date values:")
        bronze_df.groupBy("_load_date").count().orderBy(F.col("_load_date").desc()).show(5, truncate=False)

    print("[debug] Latest Bronze rows:")
    bronze_df.orderBy(F.col("_load_date").desc_nulls_last()).show(5, truncate=False)

    silver_df = spark.read.format("delta").load(silver_path)
    print(f"[debug] Silver current rows: {silver_df.count():,}")
    silver_df.show(5, truncate=False)

In [0]:
# Cell 5: Orchestration logic
def run_silver_orchestration():
    print("[INFO] Orchestration started.")
    setup_widgets()
    TABLE_NAME, TABLE_TYPE, LOAD_DATE, STORAGE_ACCOUNT_VALUE, IS_INCREMENTAL = get_params()
    global STORAGE_ACCOUNT
    STORAGE_ACCOUNT = STORAGE_ACCOUNT_VALUE
    print_header(TABLE_NAME, TABLE_TYPE, LOAD_DATE, STORAGE_ACCOUNT, IS_INCREMENTAL)

    # Auth context
    try:
        print("[INFO] Loading authentication context notebook...")
        ensure_auth_context()
    except Exception as e:
        print(f"[ERROR] Auth context notebook failed: {e}")
        error_payload = {"status": "FAILURE", "step": "auth", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    required_functions = ["read_bronze", "drop_lineage_columns", "clean", "add_silver_audit", "write_silver"]
    missing_functions = [name for name in required_functions if name not in globals()]
    if missing_functions:
        print(f"[ERROR] Specialist helpers are not loaded in this session: {missing_functions}")
        error_payload = {"status": "FAILURE", "step": "helpers_missing", "error": ",".join(missing_functions)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return
    print(f"[INFO] Specialist helpers ready: {required_functions}")

    # Read from Bronze
    try:
        print("[INFO] Reading from Bronze layer...")
        raw_df = read_bronze(TABLE_NAME, TABLE_TYPE, LOAD_DATE, IS_INCREMENTAL)
        bronze_filtered_count = raw_df.count()
        print(f"[INFO] Bronze read successful. Filtered Bronze rows: {bronze_filtered_count:,}")
        if bronze_filtered_count == 0:
            print("[WARN] No Bronze rows matched the requested load_date after filtering.")
    except Exception as e:
        print(f"[ERROR] Failed to read bronze: {e}")
        error_payload = {"status": "FAILURE", "step": "read_bronze", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    # Drop Bronze lineage columns
    try:
        print("[INFO] Dropping Bronze lineage columns...")
        raw_df = drop_lineage_columns(raw_df)
        print("[INFO] Lineage columns dropped.")
    except Exception as e:
        print(f"[ERROR] Failed to drop lineage columns: {e}")
        error_payload = {"status": "FAILURE", "step": "drop_lineage", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    # Clean + type cast
    try:
        print("[INFO] Cleaning and type casting data...")
        clean_df = clean(raw_df, TABLE_NAME)
        clean_count = clean_df.count()
        print(f"\n[silver_loader] Clean rows: {clean_count:,}")
        print("[INFO] Data cleaning successful.")
    except Exception as e:
        print(f"[ERROR] Failed to clean data: {e}")
        error_payload = {"status": "FAILURE", "step": "clean", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    # Add Silver audit column
    try:
        print("[INFO] Adding Silver audit column...")
        clean_df = add_silver_audit(clean_df)
        audited_count = clean_df.count()
        print(f"[INFO] Silver audit column added. Audited rows: {audited_count:,}")
    except Exception as e:
        print(f"[ERROR] Failed to add silver audit column: {e}")
        error_payload = {"status": "FAILURE", "step": "audit", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    # Write to Silver Delta
    try:
        print("[INFO] Writing to Silver Delta table...")
        final_count = write_silver(clean_df, TABLE_NAME, IS_INCREMENTAL)
        print(f"[INFO] Write to Silver Delta successful. Final Silver rows: {final_count:,}")
    except Exception as e:
        print(f"[ERROR] Failed to write silver: {e}")
        error_payload = {"status": "FAILURE", "step": "write_silver", "error": str(e)}
        dbutils.notebook.exit(json.dumps(error_payload))
        return

    # Confirm + exit
    print(f"\n{'='*60}")
    print(f"  Silver complete")
    print(f"  Table      : {TABLE_NAME}")
    print(f"  Load date  : {LOAD_DATE}")
    print(f"  Silver rows: {final_count:,}")
    print(f"{'='*60}")

    success_payload = {
        "status": "SUCCESS",
        "table_name": TABLE_NAME,
        "silver_rows": final_count
    }
    print("[INFO] Orchestration complete. Exiting notebook.")
    dbutils.notebook.exit(json.dumps(success_payload))

In [0]:
# Cell 6: Entry point
run_silver_orchestration()